In [1]:
from modules import Train,HeadClassifierCLIPModel,CLIPExtractor,CLIPCollateFunction,CreationClipDataset,CreationProcessedDataset,creation_dataframe
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor,CLIPImageProcessor,CLIPTokenizerFast,CLIPModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


In [2]:
#Creation of the dataframes
train_df=creation_dataframe("../data/train.jsonl")

val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
#Creation of the first Clip Datasets to get the texts and images embeddings through the pretrained clip model
train_clip_dataset=CreationClipDataset(train_df)
val_clip_dataset=CreationClipDataset(val_df)

In [4]:
#Initialisation of Clip Processors
#text_processor=CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
#image_processor=CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
#Initialisation of the clip model, the device, and the batch size
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32

In [6]:
clip_collate_object=CLIPCollateFunction(processor)

In [7]:
#Creation of the first Clip Dataloaders to get the texts and images embeddings through the pretrained clip model
train_clip_dataloader=DataLoader(train_clip_dataset,batch_size=batch_size,shuffle=False,collate_fn=clip_collate_object.collate_fn)
val_clip_dataloader=DataLoader(val_clip_dataset,batch_size=batch_size,shuffle=False,collate_fn=clip_collate_object.collate_fn)

In [8]:
clip_extractor=CLIPExtractor(clip_model,device)

In [9]:
#Extraction of the pretrained CLIP embeddings (texts embeddings, images embeddings, and similarity scores)
#final_train_data=clip_extractor.get_embeddings(train_clip_dataloader,"train","./modules/clip_embeddings")
#final_val_data=clip_extractor.get_embeddings(val_clip_dataloader,"val","./modules/clip_embeddings")

In [10]:
train_data=torch.load("./modules/clip_embeddings/train_clip_embeddings.pt")
val_data=torch.load("./modules/clip_embeddings/val_clip_embeddings.pt")

In [11]:
train_data["sim_scores"]=train_data["sim_scores"].view(-1,1)
print(train_data["sim_scores"].shape)

torch.Size([8500, 1])


In [12]:
#Creation of final datasets and dataloaders for the training
train_dataset=CreationProcessedDataset(train_data)
val_dataset=CreationProcessedDataset(val_data)
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=32,shuffle=True,drop_last=True)

In [13]:
model=HeadClassifierCLIPModel(fc_layer_sizes=[512])

In [14]:
#Use of the class weights to compensate imabalances of the dataset and make more accurate predictions

class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [15]:
#Training hyperparameters
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)

In [16]:
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)

In [17]:
trainer.run_training(train_dataloader,val_dataloader,"./modules/train_savings")

2026-03-12 14:21:27.103 | INFO     | modules.train:run_training:115 - Epoch 0 :
2026-03-12 14:21:29.226 | INFO     | modules.train:run_training:185 - Epoch 0: Train Loss = 0.7215837636083927
2026-03-12 14:21:29.226 | INFO     | modules.train:run_training:186 - Epoch 0: Train Accuracy = 0.46639150943396224
2026-03-12 14:21:29.226 | INFO     | modules.train:run_training:187 - Epoch 0: Train F1 = 0.46781498714704556
2026-03-12 14:21:29.226 | INFO     | modules.train:run_training:189 - Epoch 0: Validation Loss = 0.6807590126991272
2026-03-12 14:21:29.226 | INFO     | modules.train:run_training:190 - Epoch 0: Validation Accuracy = 0.5291666666666667
2026-03-12 14:21:29.226 | INFO     | modules.train:run_training:191 - Epoch 0: Validation F1 = 0.4988240740740741
2026-03-12 14:21:29.242 | INFO     | modules.train:run_training:115 - Epoch 1 :
2026-03-12 14:21:31.234 | INFO     | modules.train:run_training:185 - Epoch 1: Train Loss = 0.6376715400308933
2026-03-12 14:21:31.235 | INFO     | modul

In [21]:
model=HeadClassifierCLIPModel(fc_layer_sizes=[512],with_scores=True)

In [22]:
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)

In [23]:
trainer.run_training(train_dataloader,val_dataloader,"./modules/train_savings_scores",with_scores=True)

2026-03-12 14:24:01.778 | INFO     | modules.train:run_training:115 - Epoch 0 :
2026-03-12 14:24:03.042 | INFO     | modules.train:run_training:185 - Epoch 0: Train Loss = 0.8291861019044552
2026-03-12 14:24:03.042 | INFO     | modules.train:run_training:186 - Epoch 0: Train Accuracy = 0.3623820754716981
2026-03-12 14:24:03.042 | INFO     | modules.train:run_training:187 - Epoch 0: Train F1 = 0.19973814061496725
2026-03-12 14:24:03.042 | INFO     | modules.train:run_training:189 - Epoch 0: Validation Loss = 0.6810848236083984
2026-03-12 14:24:03.042 | INFO     | modules.train:run_training:190 - Epoch 0: Validation Accuracy = 0.49375
2026-03-12 14:24:03.042 | INFO     | modules.train:run_training:191 - Epoch 0: Validation F1 = 0.32641213389121343
2026-03-12 14:24:03.066 | INFO     | modules.train:run_training:115 - Epoch 1 :
2026-03-12 14:24:04.409 | INFO     | modules.train:run_training:185 - Epoch 1: Train Loss = 0.8314678426058787
2026-03-12 14:24:04.409 | INFO     | modules.train:ru